# Reading from the silver table/s

In [0]:
df_cust = spark.read.table("workspace.silver.crm_cust_info")
df_az12 = spark.read.table("workspace.silver.erp_cust_az12")
df_a101 = spark.read.table("workspace.silver.erp_loc_a101")

# Creating temporary view

In [0]:
# Register them as Temporary Views so Spark SQL can see them
df_cust.createOrReplaceTempView("ci")
df_az12.createOrReplaceTempView("cb")
df_a101.createOrReplaceTempView("cl")

# Create customers dimension table

In [0]:
query = """
    SELECT
    customer_surrogate_key,
    customer_id,
    customer_key,
    firstname,
    lastname,
    birth_date,
    gender,
    country,
    marital_status,
    create_date
    FROM
        (SELECT
            ROW_NUMBER() OVER(ORDER BY customer_id) AS customer_surrogate_key,
            ci.customer_id,
            ci.customer_key,
            ci.firstname,
            ci.lastname,
            ci.marital_status,
            ci.create_date,
            cb.birth_date,
            CASE 
                WHEN ci.gender IS NULL THEN COALESCE(cb.gender, 'N/A')
                ELSE ci.gender
            END AS gender,
            cl.country
        FROM ci
        LEFT JOIN cb
        ON ci.customer_key = cb.customer_key
        LEFT JOIN cl
        ON ci.customer_key = cl.customer_key)t

"""

df = spark.sql(query)


# Write to Gold

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("gold.dim_customers")
)

In [0]:
%sql
select * from workspace.gold.dim_customers